# SCALE Lab 25 Project 3 LLM: Character Transformer Baseline

Objective: train a character-level Transformer language model on the provided Lu Xun corpus and track validation perplexity.

Competition constraints from Kaggle:

- Architecture: Transformer.
- Layers: at most 6.
- Hidden dimension: at most 512.
- Attention heads: at most 8.
- Total parameters: at most 50M.
- Vocabulary size: at most 8k.
- Tokenizer: character-level tokenizer built from unique characters and punctuation in the training text.
- Metric: perplexity (PPL).


## Plan

1. Load `train.csv` and `test.csv` from the Kaggle input directory.
2. Infer the text column, then build a character vocabulary from the training text.
3. Train a compact causal Transformer baseline.
4. Report validation loss and perplexity.
5. Save model/tokenizer artifacts and generate optional test diagnostics.

The fetched Kaggle metadata did not include a sample submission file, so this notebook focuses on a reproducible model baseline and artifact generation. The final cell writes `submission.csv` only if a clear target-like column is present in `test.csv`.


In [1]:
!uv !pip -q install deep-translator


error: unrecognized subcommand '!pip'

  tip: a similar subcommand exists: 'pip'

Usage: uv [OPTIONS] <COMMAND>

For more information, try '--help'.


In [2]:
from __future__ import annotations

import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

KAGGLE_INPUT = Path('/kaggle/input/scale-lab-25-project-3-llm')
LOCAL_INPUT = Path('../data/scale-lab-25-project-3-llm')
DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_INPUT
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)


Using device: cuda
Data directory: /kaggle/input/scale-lab-25-project-3-llm
Output directory: /kaggle/working


In [4]:
@dataclass
class Config:
    context_length: int = 128
    d_model: int = 256
    n_heads: int = 4
    n_layers: int = 4
    dim_feedforward: int = 1024
    dropout: float = 0.10
    batch_size: int = 64
    epochs: int = 3
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    train_fraction: float = 0.95
    max_vocab_size: int = 8000
    # Set these to a small integer for quick notebook smoke tests.
    max_train_batches: int | None = None
    max_valid_batches: int | None = None

cfg = Config()
asdict(cfg)


{'context_length': 128,
 'd_model': 256,
 'n_heads': 4,
 'n_layers': 4,
 'dim_feedforward': 1024,
 'dropout': 0.1,
 'batch_size': 64,
 'epochs': 3,
 'learning_rate': 0.0003,
 'weight_decay': 0.01,
 'grad_clip': 1.0,
 'train_fraction': 0.95,
 'max_vocab_size': 8000,
 'max_train_batches': None,
 'max_valid_batches': None}

## Load Data

The notebook infers the main text column by selecting the object/string column with the largest total text length. If the competition schema is different, set `TEXT_COLUMN` manually after inspecting the printed columns.


In [5]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'Missing train.csv at {TRAIN_PATH}. On Kaggle, add the competition dataset to the notebook.')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else pd.DataFrame()

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
display(train_df.head())
if not test_df.empty:
    display(test_df.head())


Train shape: (2867, 3)
Test shape: (100, 3)
Train columns: ['name', 'type', 'context']
Test columns: ['name', 'type', 'context']


,name,type,context
0,19351223①致李小峰,书信,\n19351223①致李小峰\n351223①致李小峰 小峰兄： 惠示诵悉。《集...
1,三论“文人相轻”,杂文,\n三论“文人相轻”\t\t\t\n\t\t《芒种》第八期上有一篇魏金枝先生的《分明的是非和...
2,19360105①致曹靖华,书信,\n19360105①致曹靖华\n360105①致曹靖华 汝珍兄： 一月一日信收到...
3,19350629①致赖少麒,书信,\n19350629①致赖少麒\n350629①致赖少麒 少麒先生： 五月二八日的...
4,19331115②致姚克,书信,\n19331115②致姚克\n331115②致姚克 Y先生： 九日函收到。《申报...


,name,type,context
0,19240226致李秉中,书信,\n19240226致李秉中\n240226 致李秉中 秉中兄： 我的时间如下，但...
1,女人未必多说谎,杂文,\n女人未必多说谎\t\t\t\n\t\t侍桁先生在《谈说谎》里，以为说谎的原因之一是由于弱...
2,19350331致母亲,书信,\n19350331致母亲\n350331致母亲 母亲大人膝下，敬禀者，廿三的信，早收到了...
3,19330803致黎烈文,书信,\n19330803致黎烈文\n330803致黎烈文 烈文先生： 得七月卅一日信，...
4,19350809致黄源,书信,\n19350809致黄源\n350809致黄源 河清先生： 五日信并《世界文库》...


In [6]:
def infer_text_column(df: pd.DataFrame) -> str:
    object_cols = [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])]
    if not object_cols:
        raise ValueError('No text-like columns found. Set TEXT_COLUMN manually.')
    scores = {c: df[c].fillna('').astype(str).str.len().sum() for c in object_cols}
    return max(scores, key=scores.get)

TEXT_COLUMN = infer_text_column(train_df)
print('Using text column:', TEXT_COLUMN)

train_texts = train_df[TEXT_COLUMN].fillna('').astype(str).tolist()
corpus = '\n'.join(train_texts)
print(f'Training texts: {len(train_texts):,}')
print(f'Corpus characters: {len(corpus):,}')
print('Preview:')
print(corpus[:500])


Using text column: context
Training texts: 2,867
Corpus characters: 2,868,407
Preview:

19351223①致李小峰
351223①致李小峰  小峰兄：     惠示诵悉。《集外集拾遗》抄出大半，尚有数篇在觅期刊，编好须在明年了。     北新以社会情形和内部关系之故，自当渐不如前，但此非我个人之力所能如何，而况我亦年渐衰迈，体力已不如前哉？区区一二本书，恐无甚效，而北新又须选择，我的作品又很不平稳，如何是好。     附笺并稿一件，乞转交赵先生。 迅顿首十二月二十三日 

三论“文人相轻”			
		《芒种》第八期上有一篇魏金枝先生的《分明的是非和热烈的好恶》，是为以前的《文学论坛》上的《再论“文人相轻”》而发的。他先给了原则上的几乎全体的赞成，说，“人应有分明的是非，和热烈的好恶，这是不错的，文人应更有分明的是非，和更热烈的好恶，这也是不错的。”中间虽说“凡人在落难时节……能与猿鹤为伍，自然最好，否则与鹿豕为伍，也是好的。即到千万没有办法的时候，至于躺在破庙角里，而与麻疯病菌为伍，倘然我的体力，尚能为自然的抗御，因而不至毁灭以死，也比被实际上也做着骗子屠夫的所诱杀脔割，较为心愿。”看起来好像有些微辞，但其实说的是他的憎恶骗子屠夫，远在猿鹤以至麻疯病菌之上，和《论坛》上


## Reading The Lu Xun Corpus

The training file is described by the competition as `提供的鲁迅文集文本`, meaning **the provided collection of Lu Xun writings**. Lu Xun (`鲁迅`, 1881-1936) is a central figure in modern Chinese literature, known for short stories and essays written in early modern Chinese.

Because this notebook is for a Kaggle competition, translations are included only as an **exploration aid**. They are not used for tokenization, training, validation, or submission generation. The competition rules say training must use only the provided data, so external translation services should not become model inputs or labels.

The next cells print many Chinese snippets from `train.csv`. If a translation package is available, they also show rough English machine translations. These translations are intended to help a non-Chinese speaker understand the corpus, not to produce polished literary translations.


In [7]:
def clean_for_reading(text: str) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text

READING_EXAMPLES = 24
SNIPPET_CHARS = 180
rng = np.random.default_rng(SEED)

non_empty_texts = [clean_for_reading(t) for t in train_texts if clean_for_reading(t)]
if not non_empty_texts:
    raise ValueError('No non-empty training texts found to display.')

sample_indices = rng.choice(len(non_empty_texts), size=min(READING_EXAMPLES, len(non_empty_texts)), replace=False)
reading_examples = []
for n, idx in enumerate(sample_indices, start=1):
    text = non_empty_texts[int(idx)]
    if len(text) > SNIPPET_CHARS:
        start = int(rng.integers(0, max(1, len(text) - SNIPPET_CHARS)))
        snippet = text[start:start + SNIPPET_CHARS]
    else:
        start = 0
        snippet = text
    reading_examples.append({'example': n, 'source_row': int(idx), 'start_char': start, 'chinese_text': snippet})

examples_df = pd.DataFrame(reading_examples)
pd.set_option('display.max_colwidth', 260)
display(examples_df)


,example,source_row,start_char,chinese_text
0,1,244,1104,没有幽默作家，大抵是讽刺作家。博人一笑的作品，汉代以来也有些，是否选入这全集？如要，我可选些给你，那是有点难译的。 迄今为日本所介绍的中国文章，大抵是较轻松易懂的东西；坚实而有趣的作品，如陶潜的《闲情赋》之类，一点也没有译。能读那类作品的汉学家，自己也写难懂的汉文，不知是想给中国人读，还是想吓吓日本人？我想这种想人未曾留意过的工作，是应该做的，但出版家怕也有
1,2,1467,267,，有时也还喝，正如还要翻翻中国书一样。但是和青年谈起饮食来，我总说：你不要喝酒。听的人虽然知道我曾经纵酒，而都明白我的意思。我即使自己出的是天然痘，决不因此反对牛痘；即使开了棺材铺，也不来讴歌瘟疫的。就是这么一个意思。还有一种顺便而不相干的声明。一个朋友告诉我，《晨报副刊》上有《评玉君》的文章，其中提起我在《民众文艺》上所载的《战士和苍蝇》的话。其实我做那篇
2,3,2785,655,有些英雄们又要责我散布黑暗，阻碍革命了。一理是也有一理的，现在易犯嫌疑，忠实同志被误解为共党，或关或释的，报上向来常见。万一不幸，沉冤莫白，那真是……。倘使常常提起这些来，也许未免会短壮士之气。但是，革命被头挂退的事是很少有的，革命的完结，大概只由于投机者的潜入。也就是内里蛀空。这并非指赤化，任何主义的革命都如此。但不是正因为黑暗，正因为没有出路，所以要革命
3,4,2174,6,05致沈雁冰 361005致沈雁冰 明甫先生： 四日信收到。 “顾问”之列，我不愿加入，因为先前为了这一类职衔，吃苦不少，而且甚至于由此发生事端，所以现在要回避了。 在十四日之前，当投稿一篇，虽然题目未能十分确定。 萧红一去之后，并未给我一信，通知地址；近闻已将回沪，然亦不知其详，所以来意不能转达也。 昨看《冰天雪地》，还好。专此布复，即请 著安。 树上十月
4,5,1988,14,30525致周茨石 茨石先生： 来信收到了。灾区的真实情形，南边的坐在家里的人，知道得很少。报上的记载，也无非是“惨不忍睹”一类的含浑文字，所以倘有切实的纪录或描写出版，是极好的。 不过商量办报和看文章，我恐怕无此时间及能力，因为我年纪大起来，家累亦重，没有这工夫了。但我的意见，以为（1）如办刊物，最好不要弄成文学杂志，而只给读者以一种诚实的材料；（2）用这
5,6,1233,224,感卷逃，连小病也不生了。偶然看看文学家的名文，说是秋花为之惨容，大海为之沉默云云，只是愈加感到自己的麻木。我就从来没有见过秋花为了我在悲哀，忽然变了颜色；只要有风，大海是总在呼啸的，不管我爱闹还是爱静。冰莹女士的佳作告诉我们：“晨是学科学的，但在这一刹那，完全忘掉了他的志趣，存在他脑海中的只有一个尽量地享受自然美景的目的。……”这也是一种福气。科学我学的很浅
6,7,523,25,白涛先生： 顷接到五月廿六信。木刻集于廿四日寄上一本，现在想已收到了罢。三四日内，当嘱书店再寄上十六本，分四包，无须用现银换取法，只要看包上所贴之邮票，平分每册邮费，加上每册若干，将来一并付还书店就好了。 同时又得铁耕兄信，谓他的旧刻木板，皆存先生处。倘此信到日，尚未回汕，则希回汕时将他的《等父亲回来》（即刻母子二人，一坐一立者）那一块一并寄下。但如来不及
7,8,2247,938,版税二百。季巿来，并赠海婴糖果二合，晚同至东亚食堂夜饭。夜雨。十四日 小雨。午后同广平携海婴去理发。往内山书店买书两本，四元二角，又《喜多川歌 》一本附图一幅(六大浮世绘师之一)，九元八角。十五日 昙。上午同广平携海婴往篠崎医院诊，并付诊疗费五十七元。十六日 昙。午后得母亲信，十二日发。得增田君信，七P19日发。下午往北新书店。往朵云轩买单宣百五十枚，特别宣
8,9,2446,115,那里的人民哭着呢还是笑着呢，我们不知道。香港似乎很太平，住在这里的中国人，舒服呢还是不很舒服呢，别人也不知道。 发表自己的思想，感情给大家知道的是要用文章的，然而拿文章来达意，现在一般的中国人还做不到。这也怪不得我们；因为那文字，先就是我们的祖先留传给我们的可怕的遗产。人们费了多年的工夫，还是难于运用。因为难，许多人便不理它了，甚至于连自己的姓也写不清是张还
9,10,1862,44,上海，先前我以为是到乡下去了。暂时“消沈”一下，也好的，算是休息休息，有了力气，自然会不“消沈”的，疲劳了还是做，必至于乏力而后已，我憎恶那些拿了鞭子，专门鞭扑别人的人们。 笔记恐怕也不见得稳当，因为无论做什么东西，气息总不会改的。见闻也有，但想起来也大抵无聊的居多，自以为可写的，又一定通不过，一时真也决不下，看将来再说罢。 《春牛图》我没有，也不知道何处可


In [8]:
def build_translator():
    try:
        from deep_translator import GoogleTranslator
        translator = GoogleTranslator(source='zh-CN', target='en')
        return lambda text: translator.translate(text)
    except Exception as exc:
        print('Machine translation is unavailable in this environment.')
        print('To enable it in a Kaggle notebook with internet access, run: !pip -q install deep-translator')
        print(f'Translator import/error detail: {type(exc).__name__}: {exc}')
        return None

TRANSLATE_EXAMPLES = True
translator = build_translator() if TRANSLATE_EXAMPLES else None

translated_rows = []
for row in reading_examples:
    chinese = row['chinese_text']
    if translator is None:
        english = '[translation unavailable: install deep-translator or use another translation service for exploration only]'
    else:
        try:
            english = translator(chinese)
        except Exception as exc:
            english = f'[translation failed: {type(exc).__name__}: {exc}]'
    translated_rows.append({**row, 'english_machine_translation': english})

translated_df = pd.DataFrame(translated_rows)
display(translated_df[['example', 'chinese_text', 'english_machine_translation']])


Machine translation is unavailable in this environment.
To enable it in a Kaggle notebook with internet access, run: !pip -q install deep-translator
Translator import/error detail: ModuleNotFoundError: No module named 'deep_translator'


,example,chinese_text,english_machine_translation
0,1,没有幽默作家，大抵是讽刺作家。博人一笑的作品，汉代以来也有些，是否选入这全集？如要，我可选些给你，那是有点难译的。 迄今为日本所介绍的中国文章，大抵是较轻松易懂的东西；坚实而有趣的作品，如陶潜的《闲情赋》之类，一点也没有译。能读那类作品的汉学家，自己也写难懂的汉文，不知是想给中国人读，还是想吓吓日本人？我想这种想人未曾留意过的工作，是应该做的，但出版家怕也有,[translation unavailable: install deep-translator or use another translation service for exploration only]
1,2,，有时也还喝，正如还要翻翻中国书一样。但是和青年谈起饮食来，我总说：你不要喝酒。听的人虽然知道我曾经纵酒，而都明白我的意思。我即使自己出的是天然痘，决不因此反对牛痘；即使开了棺材铺，也不来讴歌瘟疫的。就是这么一个意思。还有一种顺便而不相干的声明。一个朋友告诉我，《晨报副刊》上有《评玉君》的文章，其中提起我在《民众文艺》上所载的《战士和苍蝇》的话。其实我做那篇,[translation unavailable: install deep-translator or use another translation service for exploration only]
2,3,有些英雄们又要责我散布黑暗，阻碍革命了。一理是也有一理的，现在易犯嫌疑，忠实同志被误解为共党，或关或释的，报上向来常见。万一不幸，沉冤莫白，那真是……。倘使常常提起这些来，也许未免会短壮士之气。但是，革命被头挂退的事是很少有的，革命的完结，大概只由于投机者的潜入。也就是内里蛀空。这并非指赤化，任何主义的革命都如此。但不是正因为黑暗，正因为没有出路，所以要革命,[translation unavailable: install deep-translator or use another translation service for exploration only]
3,4,05致沈雁冰 361005致沈雁冰 明甫先生： 四日信收到。 “顾问”之列，我不愿加入，因为先前为了这一类职衔，吃苦不少，而且甚至于由此发生事端，所以现在要回避了。 在十四日之前，当投稿一篇，虽然题目未能十分确定。 萧红一去之后，并未给我一信，通知地址；近闻已将回沪，然亦不知其详，所以来意不能转达也。 昨看《冰天雪地》，还好。专此布复，即请 著安。 树上十月,[translation unavailable: install deep-translator or use another translation service for exploration only]
4,5,30525致周茨石 茨石先生： 来信收到了。灾区的真实情形，南边的坐在家里的人，知道得很少。报上的记载，也无非是“惨不忍睹”一类的含浑文字，所以倘有切实的纪录或描写出版，是极好的。 不过商量办报和看文章，我恐怕无此时间及能力，因为我年纪大起来，家累亦重，没有这工夫了。但我的意见，以为（1）如办刊物，最好不要弄成文学杂志，而只给读者以一种诚实的材料；（2）用这,[translation unavailable: install deep-translator or use another translation service for exploration only]
5,6,感卷逃，连小病也不生了。偶然看看文学家的名文，说是秋花为之惨容，大海为之沉默云云，只是愈加感到自己的麻木。我就从来没有见过秋花为了我在悲哀，忽然变了颜色；只要有风，大海是总在呼啸的，不管我爱闹还是爱静。冰莹女士的佳作告诉我们：“晨是学科学的，但在这一刹那，完全忘掉了他的志趣，存在他脑海中的只有一个尽量地享受自然美景的目的。……”这也是一种福气。科学我学的很浅,[translation unavailable: install deep-translator or use another translation service for exploration only]
6,7,白涛先生： 顷接到五月廿六信。木刻集于廿四日寄上一本，现在想已收到了罢。三四日内，当嘱书店再寄上十六本，分四包，无须用现银换取法，只要看包上所贴之邮票，平分每册邮费，加上每册若干，将来一并付还书店就好了。 同时又得铁耕兄信，谓他的旧刻木板，皆存先生处。倘此信到日，尚未回汕，则希回汕时将他的《等父亲回来》（即刻母子二人，一坐一立者）那一块一并寄下。但如来不及,[translation unavailable: install deep-translator or use another translation service for exploration only]
7,8,版税二百。季巿来，并赠海婴糖果二合，晚同至东亚食堂夜饭。夜雨。十四日 小雨。午后同广平携海婴去理发。往内山书店买书两本，四元二角，又《喜多川歌 》一本附图一幅(六大浮世绘师之一)，九元八角。十五日 昙。上午同广平携海婴往篠崎医院诊，并付诊疗费五十七元。十六日 昙。午后得母亲信，十二日发。得增田君信，七P19日发。下午往北新书店。往朵云轩买单宣百五十枚，特别宣,[translation unavailable: install deep-translator or use another translation service for exploration only]
8,9,那里的人民哭着呢还是笑着呢，我们不知道。香港似乎很太平，住在这里的中国人，舒服呢还是不很舒服呢，别人也不知道。 发表自己的思想，感情给大家知道的是要用文章的，然而拿文章来达意，现在一般的中国人还做不到。这也怪不得我们；因为那文字，先就是我们的祖先留传给我们的可怕的遗产。人们费了多年的工夫，还是难于运用。因为难，许多人便不理它了，甚至于连自己的姓也写不清是张还,[translation unavailable: install deep-translator or use another translation service for exploration only]
9,10,上海，先前我以为是到乡下去了。暂时“消沈”一下，也好的，算是休息休息，有了力气，自然会不“消沈”的，疲劳了还是做，必至于乏力而后已，我憎恶那些拿了鞭子，专门鞭扑别人的人们。 笔记恐怕也不见得稳当，因为无论做什么东西，气息总不会改的。见闻也有，但想起来也大抵无聊的居多，自以为可写的，又一定通不过，一时真也决不下，看将来再说罢。 《春牛图》我没有，也不知道何处可,[translation unavailable: install deep-translator or use another translation service for exploration only]


## Character Tokenizer

The tokenizer is deliberately simple: one Unicode character per token. It is fitted only on the training corpus, with `<pad>` and `<unk>` reserved for batching and unseen characters.


In [9]:
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN]

chars = sorted(set(corpus))
max_chars = cfg.max_vocab_size - len(SPECIAL_TOKENS)
if len(chars) > max_chars:
    raise ValueError(f'Character vocabulary has {len(chars)} unique chars, exceeding limit {max_chars}.')

itos = SPECIAL_TOKENS + chars
stoi = {ch: i for i, ch in enumerate(itos)}
pad_id = stoi[PAD_TOKEN]
unk_id = stoi[UNK_TOKEN]

print(f'Vocabulary size: {len(itos):,}')
print('First 100 vocab entries:', itos[:100])
assert len(itos) <= cfg.max_vocab_size

def encode(text: str) -> list[int]:
    return [stoi.get(ch, unk_id) for ch in text]

def decode(ids: list[int]) -> str:
    return ''.join(itos[i] for i in ids if i >= len(SPECIAL_TOKENS))

encoded = np.array(encode(corpus), dtype=np.int64)
print('Encoded length:', len(encoded))


Vocabulary size: 7,636
First 100 vocab entries: ['<pad>', '<unk>', '\t', '\n', '\r', ' ', '!', '"', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '\xa0', '§', '°', '·', '×', 'á', 'ä']
Encoded length: 2868407


In [10]:
split_idx = int(len(encoded) * cfg.train_fraction)
train_ids = encoded[:split_idx]
valid_ids = encoded[max(0, split_idx - cfg.context_length):]

print(f'Train tokens: {len(train_ids):,}')
print(f'Valid tokens: {len(valid_ids):,}')

class CharBlockDataset(Dataset):
    def __init__(self, token_ids: np.ndarray, context_length: int):
        if len(token_ids) <= context_length + 1:
            raise ValueError('Not enough tokens for the configured context length.')
        self.token_ids = token_ids
        self.context_length = context_length

    def __len__(self) -> int:
        return len(self.token_ids) - self.context_length - 1

    def __getitem__(self, idx: int):
        chunk = self.token_ids[idx: idx + self.context_length + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

train_ds = CharBlockDataset(train_ids, cfg.context_length)
valid_ds = CharBlockDataset(valid_ids, cfg.context_length)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
valid_loader = DataLoader(valid_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, y_batch.shape)
print(decode(x_batch[0].tolist()[:120]))


Train tokens: 2,724,986
Valid tokens: 143,549
torch.Size([64, 128]) torch.Size([64, 128])
 四月十七日漫吾輩は猫でぁる一本  ０·三０英文学散策一本  二·四０两地书二十本  一四·００  四月十九日一立斎重一本  六·００  四月二十日插画本十月一本  靖华寄来  四月二十一日人生十字路一本  一·六０  四月二十二日世界


## Model

This is a causal Transformer encoder used as a decoder-only language model. The causal mask prevents positions from attending to future characters.


In [11]:
class CausalTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, config: Config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.context_length, config.d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.n_heads,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=config.n_layers)
        self.norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=input_ids.device, dtype=torch.bool), diagonal=1)
        x = self.blocks(x, mask=causal_mask)
        x = self.norm(x)
        return self.lm_head(x)

model = CausalTransformerLM(len(itos), cfg).to(device)
param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {param_count:,}')
print(f'Trainable parameters: {trainable_count:,}')
assert cfg.n_layers <= 6
assert cfg.d_model <= 512
assert cfg.n_heads <= 8
assert param_count <= 50_000_000


Total parameters: 5,147,136
Trainable parameters: 5,147,136


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [12]:
def run_epoch(loader: DataLoader, train: bool) -> float:
    model.train(train)
    total_loss = 0.0
    total_tokens = 0
    max_batches = cfg.max_train_batches if train else cfg.max_valid_batches

    for step, (x, y) in enumerate(loader, start=1):
        if max_batches is not None and step > max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            scheduler.step()

        token_count = y.numel()
        total_loss += loss.item() * token_count
        total_tokens += token_count

    return total_loss / max(1, total_tokens)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
total_steps = len(train_loader) * cfg.epochs if cfg.max_train_batches is None else cfg.max_train_batches * cfg.epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_steps))

history = []
best_valid_loss = float('inf')
best_model_path = OUTPUT_DIR / 'char_transformer_baseline.pt'

for epoch in range(1, cfg.epochs + 1):
    start = time.time()
    train_loss = run_epoch(train_loader, train=True)
    valid_loss = run_epoch(valid_loader, train=False)
    valid_ppl = math.exp(min(20, valid_loss))
    elapsed = time.time() - start

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'valid_loss': valid_loss,
        'valid_ppl': valid_ppl,
        'seconds': elapsed,
    }
    history.append(row)
    print(row)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
display(history_df)
print('Best validation PPL:', math.exp(min(20, best_valid_loss)))
print('Saved model:', best_model_path)


{'epoch': 1, 'train_loss': 4.030349394889546, 'valid_loss': 3.595630324965756, 'valid_ppl': 36.4386609484, 'seconds': 2789.4616277217865}
{'epoch': 2, 'train_loss': 3.301452173556949, 'valid_loss': 3.566722990881032, 'valid_ppl': 35.40039544645084, 'seconds': 2781.7700362205505}
{'epoch': 3, 'train_loss': 3.1941132728993136, 'valid_loss': 3.566462756659492, 'valid_ppl': 35.39118425068579, 'seconds': 2779.9145715236664}


,epoch,train_loss,valid_loss,valid_ppl,seconds
0,1,4.030349,3.595630,36.438661,2789.461628
1,2,3.301452,3.566723,35.400395,2781.770036
2,3,3.194113,3.566463,35.391184,2779.914572


Best validation PPL: 35.39118425068579
Saved model: /kaggle/working/char_transformer_baseline.pt


## Qualitative Sampling

Sampling is not used for scoring perplexity, but it is useful for catching obvious training failures.


In [13]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 300, temperature: float = 0.9, top_k: int = 50) -> str:
    model.eval()
    ids = encode(prompt)
    if not ids:
        ids = [unk_id]
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        context = input_ids[:, -cfg.context_length:]
        logits = model(context)[:, -1, :] / max(temperature, 1e-6)
        logits[:, pad_id] = -float('inf')
        if top_k is not None and top_k > 0:
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            threshold = values[:, [-1]]
            logits = torch.where(logits < threshold, torch.full_like(logits, -float('inf')), logits)
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return decode(input_ids[0].tolist())

prompt = corpus[: min(80, len(corpus))]
print(generate(prompt, max_new_tokens=300))



19351223①致李小峰
351223①致李小峰  小峰兄：     惠示诵悉。《集外集拾遗》抄出大半，尚有数篇在觅期刊，编好须在明年了。     北新以社会广州，先用了几个编辑者先生来访，所以我现在一定付印印花定的，但倘费几千，自然恐怕不能卖钱，其余者须改正。     《集外集》已出版，不能做。现在想做一点序，这是大约尚未能够出版。     专此布复，即颂 时绥。 迅顿首上十二月二十二日 

19390401致杜衡
350401致杜衡  杜衡先生：     我奉上《博古牌子》及《新潮》，早收到了。     《现代》我想还未必能出下，但须看后再看。《译文》，也许可以从别处觅一方式，书店中购寄来，不能用，但他们不要用，而因为没有一本，所以可以取其一也。     专此布复，并请 撰安。 鲁迅　上　十二月二十七夜 

19320808致李小峰
3208


## Save Artifacts

Artifacts include tokenizer mappings, model configuration, training history, and the best model weights.


In [14]:
tokenizer_path = OUTPUT_DIR / 'char_tokenizer.json'
config_path = OUTPUT_DIR / 'char_transformer_config.json'
history_path = OUTPUT_DIR / 'training_history.csv'

with tokenizer_path.open('w', encoding='utf-8') as f:
    json.dump({'itos': itos, 'pad_token': PAD_TOKEN, 'unk_token': UNK_TOKEN, 'text_column': TEXT_COLUMN}, f, ensure_ascii=False, indent=2)

with config_path.open('w', encoding='utf-8') as f:
    json.dump({**asdict(cfg), 'vocab_size': len(itos), 'param_count': param_count}, f, ensure_ascii=False, indent=2)

history_df.to_csv(history_path, index=False)
print('Saved:', tokenizer_path)
print('Saved:', config_path)
print('Saved:', history_path)


Saved: /kaggle/working/char_tokenizer.json
Saved: /kaggle/working/char_transformer_config.json
Saved: /kaggle/working/training_history.csv


## Optional Test Diagnostics / Submission Stub

Without a sample submission file, the safest output is test perplexity diagnostics when `test.csv` contains a text column. If the competition later provides a required submission schema, adapt this cell to that schema.


In [15]:
submission_path = OUTPUT_DIR / 'submission.csv'

if test_df.empty:
    print('No test.csv found; skipping submission output.')
else:
    test_text_column = TEXT_COLUMN if TEXT_COLUMN in test_df.columns else infer_text_column(test_df)
    test_texts = test_df[test_text_column].fillna('').astype(str).tolist()

    # If test rows contain full text, compute each row negative log-likelihood as a diagnostic.
    @torch.no_grad()
    def text_nll(text: str) -> tuple[float, float]:
        ids = encode(text)
        if len(ids) <= 1:
            return float('nan'), float('nan')
        losses = []
        for start in range(0, len(ids) - 1, cfg.context_length):
            chunk = ids[start: start + cfg.context_length + 1]
            if len(chunk) <= 1:
                continue
            x = torch.tensor(chunk[:-1], dtype=torch.long, device=device).unsqueeze(0)
            y = torch.tensor(chunk[1:], dtype=torch.long, device=device).unsqueeze(0)
            logits = model(x)
            loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), reduction='mean')
            losses.append(loss.item())
        mean_nll = float(np.mean(losses)) if losses else float('nan')
        return mean_nll, math.exp(min(20, mean_nll)) if math.isfinite(mean_nll) else float('nan')

    rows = []
    for idx, text in enumerate(test_texts):
        nll, ppl = text_nll(text)
        rows.append({'row_id': idx, 'nll': nll, 'ppl': ppl})

    diagnostics = pd.DataFrame(rows)
    diagnostics.to_csv(OUTPUT_DIR / 'test_diagnostics.csv', index=False)
    display(diagnostics.head())
    print('Saved test diagnostics:', OUTPUT_DIR / 'test_diagnostics.csv')

    id_col = 'id' if 'id' in test_df.columns else None
    target_like_cols = [c for c in test_df.columns if c.lower() in {'target', 'label', 'answer'}]
    if id_col and target_like_cols:
        # Conservative placeholder: keep required ids and emit row-level perplexity as prediction.
        # Replace this once Kaggle exposes the exact submission schema.
        sub = pd.DataFrame({'id': test_df[id_col], target_like_cols[0]: diagnostics['ppl']})
        sub.to_csv(submission_path, index=False)
        print('Saved tentative submission:', submission_path)
    else:
        print('No unambiguous submission schema found. Adapt this cell after checking the competition sample submission or leaderboard instructions.')


,row_id,nll,ppl
0,0,1.628329,5.095355
1,1,3.696090,40.289479
2,2,2.579228,13.186948
3,3,3.177456,23.985654
4,4,2.731601,15.357453


Saved test diagnostics: /kaggle/working/test_diagnostics.csv
No unambiguous submission schema found. Adapt this cell after checking the competition sample submission or leaderboard instructions.


## Notes

- Increase `epochs` for a real run after confirming the notebook executes end-to-end.
- Try `d_model=384`, `n_layers=6`, and `n_heads=6` or `8` if Kaggle runtime allows and the parameter count remains below 50M.
- Longer `context_length` can improve language modeling but increases memory use quadratically in attention.
